In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn
import jax
import matplotlib.pyplot as plt
import numpy as onp
import equinox as eqx
jax.config.update("jax_enable_x64", True)
from functools import partial
import pyvista as pv

import lineax as lx
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver, MultiGroupOperatorScanCheckpoint
from jax_sn.solution_domain import SolutionDomainData
from typing import Tuple, Callable, Iterable
from dataclasses import dataclass
from breeding_blanket_setup import _get_bb_args, _setup_bb_mgop_symmetric, aux_data_dict, bb_data_dict


In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver, MultiGroupOperator
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan
from jax_sn.parametrisation.multigroup_parametrisation import MultigroupOperatorParametrisedObjective, MultigroupOperatorParametrisedVerticesNoToposort
from jax_sn.tools.parametrisation_tools import ParametrisationValues, ParametrisedModuleTheta
from jax_sn.operators.transport_operator import TransportOperatorVmapPallas, TransportOperatorMultipleAnglesVmap, TransportOperatorBatchedPallas, TransportOperatorVmap
import mcbb
from mcbb.linear_blanket import OpenMCLinearBlanketSimulation, OpenMCSymmetricLinearBlanketSimulation
import openmc
from functools import lru_cache

In [ ]:

def create_openmc_data(blanket : mcbb.linear_blanket.Blanket1D, groups_list : list, settings : mcbb.blanket_base.OpenMCSettingsBlanket, l_order, n_x_tally = 200) -> Tuple[dict, dict, dict]:
    simulation_symm = OpenMCSymmetricLinearBlanketSimulation(blanket, settings, n_x_tally, groups_list)
    mgxs_lib, tally_result_dict =simulation_symm.create_mgxs_library(onp.flip(groups_list), l_order)
    mgxs_lib_new = mcbb.openmc_base_functions.mgxs_lib_to_standard_formatting(mgxs_lib)
    new_bb_data_dict = {key : (mgxs_lib_new[key]['total'], mgxs_lib_new[key]['nu-scatter matrix']) for key in bb_data_dict().keys()}
    new_aux_data_dict = {key : {'nt' : mgxs_lib_new[key]['(n,Xt)']} for key in aux_data_dict().keys()}
    return new_bb_data_dict, new_aux_data_dict, tally_result_dict

@lru_cache(maxsize=  4)
def _get_hcpb_data_openmc(d_elem_regions : Iterable[float], l_order : int, n_samples_per_batch : int, groups_list : Iterable[float]) -> Tuple[dict, dict, dict]:
    # Returns bb_data_dict, aux_data_dict, tally_result_dict
    hcpb_blanket = mcbb.examples.hcpb()
    for i, layer in enumerate(hcpb_blanket.layers):
        if i == 0:
            layer.thickness = 0.5 * d_elem_regions[i]
        else:
            layer.thickness = d_elem_regions[i]
    settings = mcbb.blanket_base.OpenMCSettingsBlanket(True, "openmc_data/", 10, n_samples_per_batch)
    return create_openmc_data(hcpb_blanket, groups_list, settings, l_order)


In [ ]:
class BZShieldValues(ParametrisationValues):
    bzshield : jnp.ndarray

class BZShieldSettings(eqx.Module):
    resolution_layers : Tuple[int]
    base_layer_size   : jnp.ndarray
    
from breeding_blanket_setup import recreate_symmetric_vertices

@eqx.filter_jit
def create_bzshield_vertices(params : BZShieldValues, settings : BZShieldSettings):
    bzshield = params.bzshield
    resolution_layers = settings.resolution_layers
    base_layer_size   = settings.base_layer_size    
    size              =   base_layer_size.at[3].set(bzshield[0]).at[6].set(bzshield[1])    
    return recreate_symmetric_vertices(size, resolution_layers)[:, None]
    
class BZShieldParametrisation(ParametrisedModuleTheta):

    additional_settings : BZShieldSettings

    def __check_init__(self):
        assert isinstance(self.base_module, MultiGroupOperator), "Base module must be a MultiGroupOperator."
        assert isinstance(self.additional_settings, BZShieldSettings), "Additional settings must be of type BZShieldSettings."
    
    @eqx.filter_jit
    def to_module(self, theta : BZShieldValues):
        return MultigroupOperatorParametrisedVerticesNoToposort.reconstruct_operator(self.base_module, create_bzshield_vertices(theta, self.additional_settings))
     
def get_solution_domain(MGOP, element):
    return jax_sn.solution_domain.SolutionDomain(MGOP.transport_operator.transport_operator.solution_domain_data, element)

def fast_neutron_flux(solution_values, Ij, energy_set_boundaries, fast_eg = [14.2e6, 1e6]):
    last_element = solution_values.shape[-2] - 1    
    last_element_solvals = solution_values[..., 0, last_element, :] #[n_energy_groups, n_basis_per_element]
    overlap_matrix       = jax_sn.energy_set.compute_overlap_matrix(energy_set_boundaries, fast_eg)
    egs_fast             = overlap_matrix @ last_element_solvals  #[1, n_basis_per_element]
    return jnp.einsum("ij, j->", egs_fast, Ij)  / jnp.sqrt(4 * jnp.pi) * 1e2

def distance(theta : BZShieldValues, settings : BZShieldSettings):
    bzshield          = theta.bzshield    
    base_layer_size   = settings.base_layer_size    
    size              = base_layer_size.at[3].set(bzshield[0]).at[6].set(bzshield[1])    
    return jnp.sum(size[1:])

def tritium_breeding_ratio(solution_values, settings :BZShieldSettings, solution_domain_data : SolutionDomainData, tritium_xs):
    bz_elements = sum(settings.resolution_layers) + sum(settings.resolution_layers[1:3]) # sum all previous layers (including plasma) and then first two actual blanket layers        
    solution_values_bz = jax.lax.dynamic_slice(solution_values[:,0, :, : ], (0, bz_elements, 0 ), (solution_values.shape[0], settings.resolution_layers[3], solution_values.shape[-1])) #[n_energy_groups, n_elements_bz, n_basis_per_element]
    tritium_xs_bz  = tritium_xs['BreedingZone']['nt']
    jacobian_dets = jax.lax.dynamic_slice(solution_domain_data.precomputed_geometric_data.vol_jacobians_det, (bz_elements,), (settings.resolution_layers[3],))  #[n_elements_bz,]
    return jnp.einsum("g, geb, b, e -> ", tritium_xs_bz, solution_values_bz, solution_domain_data.fem_matrices.integration_coeff, jacobian_dets) / jnp.sqrt(4 * jnp.pi) / settings.base_layer_size[0] * 2.0 # / jnp.sqrt(4 * jnp.pi) to account for flux to scalar flux, / base_layer_size[0] to account for normalization over area, *2 for symmetry

class BZShieldMultiGroupObjective(MultigroupOperatorParametrisedObjective):
    
    aux_data              : dict
    energy_set_boundaries : Tuple[float]
    tbr_constraint        : jnp.ndarray
    ffn_constraint        : jnp.ndarray

    @eqx.filter_jit
    def objective_and_constraints(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value       
        jax.debug.print("theta {theta}", theta = theta.bzshield) 
        return jnp.array([self.distance(theta), 
                tritium_breeding_ratio(solution_values, self.theta_parametrised_module.additional_settings, mgop_theta.transport_operator.transport_operator.solution_domain_data, self.aux_data) - self.tbr_constraint,
                - self.ffn_constraint - jnp.log10(fast_neutron_flux(solution_values, mgop_theta.transport_operator.solution_domain_data.fem_matrices.integration_coeff, self.energy_set_boundaries))])

    def tritium_breeding_ratio(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value
        return tritium_breeding_ratio(solution_values, self.theta_parametrised_module.additional_settings, mgop_theta.transport_operator.transport_operator.solution_domain_data, self.aux_data)
    
    def edge_fast_flux(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value
        return fast_neutron_flux(solution_values, mgop_theta.transport_operator.solution_domain_data.fem_matrices.integration_coeff, self.energy_set_boundaries)
    
    def distance(self, theta : BZShieldValues):
        return distance(theta, self.theta_parametrised_module.additional_settings)

In [ ]:
from jax_sn.quadrature_set.quadrature_set_base import QuadratureSetReduced


def mgop_base(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, bb_data_dict):
    MGOP, source_total, _= _setup_bb_mgop_symmetric(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order,  bb_data_dict)
    from jax_sn.operators.multi_group_operator.multi_group_operator_creation import create_multi_group_operator
    if (dim == 1) or (dim ==2):
        qset = QuadratureSetReduced.from_quadrature_set(MGOP.moment_operator.quadrature_set, dim)
    else:
        qset = MGOP.moment_operator.quadrature_set
    mgop_cp = create_multi_group_operator(get_solution_domain(MGOP, get_element(dim, degree)),qset , multi_group_operator_cls=MultiGroupOperatorScanCheckpoint, single_angle_transport_operator_cls=TransportOperatorVmap)    
    source_all_eg   = jnp.stack([source_total[:qset.n_angles] if i==0 else jnp.zeros_like(source_total[:qset.n_angles]) for i in range(MGOP.n_energy_groups)])
    solver          = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(1e-12,1e-12, max_steps= 100))
    return mgop_cp, source_all_eg, solver

def setup_multigroup_objective(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, bb_data_dict, aux_data_dict, groups_list, tbr_constraint= 1.3, ffn_constraint=5.0):
    MGOPcp, source_all_eg, solver = mgop_base(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, bb_data_dict)
    additional_settings      = BZShieldSettings(tuple(n_elem_regions), jnp.array(d_elem_regions))
    operator_parametrisation = BZShieldParametrisation(MGOPcp, additional_settings)    
    return BZShieldMultiGroupObjective(operator_parametrisation, source_all_eg, solver , aux_data= aux_data_dict, energy_set_boundaries= tuple(groups_list), tbr_constraint= jnp.array(tbr_constraint), ffn_constraint= jnp.array(ffn_constraint))

def get_element(dim,degree):
    return jax_sn.solution_domain.BasixLagrangianSimplex(degree, dim)


In [ ]:
# BASE Parameters
dim = 1 
degree = 7
tn_order = 3
n_elem_regions = [5, 1, 1, 10, 5, 1, 10, 2]
d_other_dim = 10.0
n_other_dim = 1
finite_element = get_element(dim, degree)

# Variable parameters
groups_list = (14.1e6, 10e6, 3e6, 1e6, 1e4, 300, 10,0)
d_elem_regions_base = (0.2, 0.002, 0.025, 0.5, 0.385, 0.06, 0.2, 0.06)
new_bb_data_dict, new_aux_data_dict, tally_result_dict = _get_hcpb_data_openmc(d_elem_regions_base, 5, 1e4, groups_list)
values_init           = BZShieldValues(jnp.array([d_elem_regions_base[3], d_elem_regions_base[6]]))
multigroup_objective  = setup_multigroup_objective(dim, degree, d_elem_regions_base, n_elem_regions, d_other_dim, n_other_dim, tn_order, new_bb_data_dict, new_aux_data_dict, groups_list, tbr_constraint= 1.3, ffn_constraint=5.0)

In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan
from jax_sn.operators.multi_group_operator.multi_group_operator_creation import create_multi_group_operator
from jax_sn.quadrature_set import QuadratureSetReduced

In [ ]:
values_init  = BZShieldValues(jnp.array([0.2,0.2]))
multigroup_objective.objective_and_constraints(values_init)

In [ ]:

from jax_sn.tools.parametrisation_tools import ParametrisedScipyInterface
jax.config.update("jax_log_compiles", False)

In [ ]:
def generate_optimized_x(multigroup_objective, tbr_constraint, ffn_constraint):
    multigroup_objective_mod = eqx.tree_at(lambda m : [m.tbr_constraint, m.ffn_constraint], multigroup_objective, (jnp.array(tbr_constraint), jnp.array(ffn_constraint)))
    scipy_parametrised_interface = ParametrisedScipyInterface(multigroup_objective_mod, (lambda x : BZShieldValues(x), lambda y : y.bzshield))
    constraints = [{"type" : "ineq", "fun" : scipy_parametrised_interface.constraints, "jac": scipy_parametrised_interface.jac_constraints},
    ]

    from scipy.optimize import minimize
    start_theta = values_init.bzshield  

    aa = minimize(fun = scipy_parametrised_interface.objective,
            jac = scipy_parametrised_interface.grad_objective,
            x0 = start_theta,
            constraints = constraints,
            method = 'SLSQP',
            options = {'disp': True, 'ftol': 1e-6, 'maxiter': 100},
            bounds = [(0.0, None), (0.0, None)]
    )
    theta_final = scipy_parametrised_interface.f_theta_parametrisation_values[0](jnp.array(aa.x))
    return aa, theta_final, multigroup_objective_mod.objective_and_constraints(theta_final) + jnp.array([0, tbr_constraint, ffn_constraint])



In [ ]:
aa = generate_optimized_x(multigroup_objective, 1.4, 6.0)

In [ ]:
def generate_pareto_front(tbr_range, eff_log_range):
    tbr_i = jnp.linspace(tbr_range[0], tbr_range[1], 20)
    eff_log_i = jnp.linspace(eff_log_range[0], eff_log_range[1], 20)
    pareto_points = []
    for tbr in tbr_i:
        for eff_log in eff_log_i:
            print(f"Optimizing for TBR constraint {tbr}, log10(FFN) constraint {eff_log}")
            aa, theta_opt, dtbreff = generate_optimized_x(multigroup_objective, tbr, eff_log)

            pareto_points.append((tbr, eff_log, dtbreff, theta_opt))
    return pareto_points

pareto_optimized = generate_pareto_front(( 1.0,1.47), (4.0,7.0))

In [ ]:
pareto_front = onp.array([p[2] for p in pareto_optimized])  # Extract dtbreff values

In [ ]:
pareto_front_negatives = onp.array([pareto_front[:, 0], -pareto_front[:, 1], -pareto_front[:, 2]]).T

In [ ]:
s_front = pareto_front_negatives[:,0].reshape(20,20)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(projection="3d")
ax.scatter(pareto_front_negatives[:,2 ], pareto_front_negatives[:, 1], pareto_front_negatives[:, 0], s = 100)
plt.xlabel("- log10(Edge Fast Flux)")
plt.ylabel(" - Tritium Breeding Ratio")
ax.set_zlabel(" Blanket Thickness [m]")
plt.tight_layout()